# Assistiv Systems — Rural Access Vulnerability Index (RAVI) v1.4
## Master Pipeline

Scores all 1,065 Kent & Medway LSOAs for rural access vulnerability.

**Run order — each cell builds on the previous:**
1. **Cell 1** — Install dependencies
2. **Cell 2** — Fetch data, calculate RAVI scores, commit JSON
3. **Cell 3** — Add precise ONS lat/lon coordinates
4. **Cell 4** — Add ONS settlement place names

**Before running:** Add `GITHUB_TOKEN` to Colab Secrets (🔑 left sidebar)

**Safe to re-run individual cells** — each loads the current JSON from GitHub, modifies only its own fields, and recommits. Coordinates are never overwritten by Cell 4. Place names are never overwritten by Cell 3.

---
*MHCLG IMD 2019 · mySociety/ONS RUC 2021 · Census 2021 Nomis · ONS LSOA Centroids 2021 · ONS BUA Lookup 2011*

## Cell 1 — Install Dependencies

In [ ]:
!pip install requests pandas openpyxl --quiet
print("Dependencies installed ✓")

## Cell 2 — Data Pipeline
Fetches IMD 2019, RUC 2021, Census 2021 → RAVI scores → GitHub.
Coordinates are grid placeholders — run Cell 3 next.

In [ ]:
"""
RAVI Cell 2 — Data pipeline: IMD 2019, RUC 2021, Census 2021 → RAVI scores → GitHub
"""
import requests, pandas as pd, json, base64, io
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()

KENT_LAD_CODES = [
    'E07000105','E07000106','E07000107','E07000108','E07000109',
    'E07000110','E06000035','E07000111','E07000112','E07000113',
    'E07000114','E07000115','E07000116',
]
WEIGHTS = {'geo_barriers':0.30,'ruc_score':0.25,'pct_65plus':0.20,'pct_no_car':0.15,'pct_llti':0.10}
assert abs(sum(WEIGHTS.values())-1.0)<0.001
LAD_NAMES = {
    'E07000105':'Ashford','E07000106':'Canterbury','E07000107':'Dartford',
    'E07000108':'Dover','E07000109':'Gravesham','E07000110':'Maidstone',
    'E06000035':'Medway','E07000111':'Sevenoaks','E07000112':'Folkestone & Hythe',
    'E07000113':'Swale','E07000114':'Thanet','E07000115':'Tonbridge & Malling',
    'E07000116':'Tunbridge Wells',
}
ENG_FALLBACKS = {'pct_65plus':18.4,'pct_no_car':25.6,'pct_llti':17.8}
HEADERS = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}
print(f"Weights: {sum(WEIGHTS.values()):.2f} ✓")

# ── IMD 2019 ──────────────────────────────────────────────────────────
print("\n── Step 1: IMD 2019 Geographic Barriers ──")
IMD_URL = ("https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/"
           "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv")
r = requests.get(IMD_URL, timeout=120, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code != 200:
    raise RuntimeError("IMD download failed. See: https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019")
imd_raw = pd.read_csv(io.StringIO(r.text))
geo_col = next((c for c in imd_raw.columns if 'Geographical Barriers' in c and 'Score' in c), None)
if not geo_col: raise ValueError(f"Geo barriers column not found. Columns: {list(imd_raw.columns)}")
print(f"  Geo barriers column: {geo_col!r}")
imd_kent = imd_raw[imd_raw['Local Authority District code (2019)'].isin(KENT_LAD_CODES)][
    ['LSOA code (2011)','LSOA name (2011)','Local Authority District code (2019)',geo_col]
].copy()
imd_kent.columns = ['lsoa_code','lsoa_name','lad_code','geo_barriers_score']
imd_kent = imd_kent.dropna(subset=['geo_barriers_score']).reset_index(drop=True)
print(f"  Kent LSOAs: {len(imd_kent)} | Range: {imd_kent.geo_barriers_score.min():.3f} – {imd_kent.geo_barriers_score.max():.3f}")

# ── RUC 2021 ──────────────────────────────────────────────────────────
print("\n── Step 2: Rural Urban Classification ──")
r = requests.get("https://raw.githubusercontent.com/mysociety/uk_ruc/main/output/composite_ruc.csv", timeout=30, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code == 200:
    ruc_raw = pd.read_csv(io.StringIO(r.text))
    ruc_col = 'ukruc-3' if 'ukruc-3' in ruc_raw.columns else 'ukruc-2'
    code_map  = {0:'Urban',1:'Rural town/village',2:'Hamlet/Isolated'} if ruc_col=='ukruc-3' else {0:'Urban',1:'Rural'}
    score_map = {0:0,1:55,2:90} if ruc_col=='ukruc-3' else {0:10,1:70}
    ruc_kent = ruc_raw[ruc_raw['lsoa'].isin(imd_kent.lsoa_code)][['lsoa',ruc_col]].copy()
    ruc_kent.columns = ['lsoa_code','ruc_raw']
    ruc_kent['ruc_label'] = ruc_kent.ruc_raw.map(code_map).fillna('Unknown')
    ruc_kent['ruc_score'] = ruc_kent.ruc_raw.map(score_map).fillna(30.0)
    imd_kent = imd_kent.merge(ruc_kent[['lsoa_code','ruc_label','ruc_score']], on='lsoa_code', how='left')
    imd_kent['ruc_label'] = imd_kent['ruc_label'].fillna('Unknown')
    imd_kent['ruc_score'] = imd_kent['ruc_score'].fillna(30.0)
    print(f"  Matched: {imd_kent.ruc_label.ne('Unknown').sum()}/{len(imd_kent)}")
    print(f"  Distribution:\n{imd_kent.ruc_label.value_counts().to_string()}")
else:
    print("  WARNING: RUC unavailable — using 30 for all")
    imd_kent['ruc_label'] = 'Unknown'; imd_kent['ruc_score'] = 30.0

# ── CENSUS 2021 ───────────────────────────────────────────────────────
print("\n── Step 3: Census 2021 via Nomis ──")
def fetch_nomis(dataset_id, params, description, col, fallback):
    url = (f"https://www.nomisweb.co.uk/api/v01/dataset/{dataset_id}.data.csv"
           f"?date=latest&geography=TYPE297&measures=20301&{params}&select=GEOGRAPHY_CODE,OBS_VALUE")
    print(f"  {description}...")
    try:
        r = requests.get(url, timeout=60, headers=HEADERS)
        print(f"    HTTP {r.status_code} | {len(r.content):,} bytes")
        if r.status_code==200 and len(r.content)>500:
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = ['lsoa_code', col]
            df = df[df.lsoa_code.isin(imd_kent.lsoa_code)]
            if len(df)>100:
                df = df.groupby('lsoa_code')[col].sum().reset_index()
                print(f"    ✓ {len(df)} LSOAs | Mean: {df[col].mean():.1f}%")
                return df
        print(f"    Using England average {fallback}%")
    except Exception as e:
        print(f"    Error: {e} — using {fallback}%")
    return None

age_df  = fetch_nomis('NM_2010_1','c2021_age_h=7,8,9,10,11,12,13,14,15,16,17,18','Age 65+','pct_65plus',ENG_FALLBACKS['pct_65plus'])
car_df  = fetch_nomis('NM_2072_1','c2021_carsvan_4=1','No car','pct_no_car',ENG_FALLBACKS['pct_no_car'])
llti_df = fetch_nomis('NM_2064_1','c2021_disability_3=1','LLTI','pct_llti',ENG_FALLBACKS['pct_llti'])
for df, col in [(age_df,'pct_65plus'),(car_df,'pct_no_car'),(llti_df,'pct_llti')]:
    if df is not None:
        imd_kent = imd_kent.merge(df, on='lsoa_code', how='left')
    else:
        imd_kent[col] = None
    imd_kent[col] = imd_kent[col].fillna(ENG_FALLBACKS[col])

# ── RAVI SCORE ────────────────────────────────────────────────────────
print("\n── Step 4: RAVI scores ──")
def norm(s):
    mn,mx = s.min(),s.max()
    return pd.Series([50.0]*len(s),index=s.index) if mx==mn else ((s-mn)/(mx-mn)*100).round(1)
imd_kent['geo_barriers_norm'] = norm(imd_kent.geo_barriers_score)
imd_kent['ruc_norm']          = norm(imd_kent.ruc_score)
imd_kent['age_norm']          = norm(imd_kent.pct_65plus)
imd_kent['car_norm']          = norm(imd_kent.pct_no_car)
imd_kent['llti_norm']         = norm(imd_kent.pct_llti)
imd_kent['ravi'] = (
    imd_kent.geo_barriers_norm*WEIGHTS['geo_barriers'] + imd_kent.ruc_norm*WEIGHTS['ruc_score'] +
    imd_kent.age_norm*WEIGHTS['pct_65plus'] + imd_kent.car_norm*WEIGHTS['pct_no_car'] +
    imd_kent.llti_norm*WEIGHTS['pct_llti']
).round(1)
def band(s): return 'critical' if s>=70 else 'high' if s>=55 else 'moderate' if s>=40 else 'low'
imd_kent['ravi_band'] = imd_kent.ravi.apply(band)
imd_kent['district']  = imd_kent.lad_code.map(LAD_NAMES)
print(f"RAVI: {imd_kent.ravi.min():.1f} – {imd_kent.ravi.max():.1f} | Mean: {imd_kent.ravi.mean():.1f}")
print(f"Risk bands:\n{imd_kent.ravi_band.value_counts().to_string()}")

# ── DISTRICT SUMMARY ──────────────────────────────────────────────────
district_summary = {}
for dist, grp in imd_kent.groupby('district'):
    high = grp[grp.ravi>=55]
    district_summary[dist] = {
        'lsoa_count':int(len(grp)), 'mean_ravi':round(float(grp.ravi.mean()),1),
        'max_ravi':round(float(grp.ravi.max()),1), 'high_ravi_count':int(len(high)),
        'pct_high':round(len(high)/len(grp)*100,1),
        'top_lsoa':str(grp.nlargest(1,'ravi').iloc[0]['lsoa_name']),
    }
print(f"\n{'District':<25} {'Mean':>5}  {'Max':>5}  {'High/Crit':>9}")
for d,s in sorted(district_summary.items(),key=lambda x:-x[1]['mean_ravi']):
    print(f"  {d:<23} {s['mean_ravi']:>5.1f}  {s['max_ravi']:>5.1f}  {s['high_ravi_count']:>9}")

# ── BUILD AND COMMIT ──────────────────────────────────────────────────
# Grid placeholder coords — Cell 3 replaces with precise ONS centroids
DC = {
    'Ashford':(51.148,0.874),'Canterbury':(51.279,1.076),'Dartford':(51.446,0.221),
    'Dover':(51.126,1.313),'Folkestone & Hythe':(51.083,1.167),'Gravesham':(51.442,0.368),
    'Maidstone':(51.272,0.523),'Medway':(51.385,0.523),'Sevenoaks':(51.272,0.187),
    'Swale':(51.333,0.753),'Thanet':(51.358,1.383),'Tonbridge & Malling':(51.197,0.273),
    'Tunbridge Wells':(51.132,0.263),
}
dc = {}
for _, row in imd_kent.iterrows():
    d = row.district
    i = dc.get(d,0); dc[d]=i+1
    c = DC.get(d,(51.27,0.52))
    imd_kent.at[_,'lat'] = round(c[0]-0.07+(i//10)*0.016,5)
    imd_kent.at[_,'lon'] = round(c[1]-0.07+(i%10)*0.016,5)

lsoa_list = [{
    'lsoa_code':row.lsoa_code,'lsoa_name':str(row.lsoa_name),
    'district':str(row.get('district','')),'lad_code':row.lad_code,
    'lat':round(float(row.get('lat',51.27)),5),'lon':round(float(row.get('lon',0.52)),5),
    'ravi':round(float(row.ravi),1),'ravi_band':row.ravi_band,'place_name':'',
    'signals':{
        'geo_barriers_score':round(float(row.geo_barriers_score),3),
        'geo_barriers_norm':round(float(row.geo_barriers_norm),1),
        'ruc_label':str(row.get('ruc_label','Unknown')),
        'ruc_score':round(float(row.get('ruc_score',30)),1),
        'ruc_norm':round(float(row.ruc_norm),1),
        'pct_65plus':round(float(row.pct_65plus),1),'age_norm':round(float(row.age_norm),1),
        'pct_no_car':round(float(row.pct_no_car),1),'car_norm':round(float(row.car_norm),1),
        'pct_llti':round(float(row.pct_llti),1),'llti_norm':round(float(row.llti_norm),1),
    }
} for _,row in imd_kent.iterrows()]

output = {
    'meta':{
        'generated':datetime.now(timezone.utc).isoformat(),
        'description':'Rural Access Vulnerability Index — Kent & Medway LSOAs',
        'version':'1.4','lsoa_count':len(lsoa_list),
        'sources':{'geo_barriers':'IMD 2019 File 7 (MHCLG)','ruc':'mySociety/ONS RUC 2021',
                   'census':'Census 2021 Nomis or England average fallback'},
        'weights':WEIGHTS,'census_source':'real' if age_df is not None else 'england_average_fallback',
        'coord_method':'grid placeholder — run Cell 3','has_coordinates':True,'has_place_names':False,
    },
    'district_summary':district_summary,'lsoas':lsoa_list,
}

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")
    return r.status_code in(200,201)

commit(output, GITHUB_FILE, GITHUB_TOKEN,
       f"RAVI v1.4 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — {len(lsoa_list)} LSOAs")
print(f"\nCell 2 done — run Cell 3 for precise coordinates")


## Cell 3 — Precise ONS Coordinates
Fetches ONS 2021 population-weighted centroids via ArcGIS geometry extraction.
Loads current JSON, updates lat/lon only, recommits.
**Run after Cell 2.**

In [ ]:
"""
RAVI Cell 3 — Add precise ONS population-weighted centroids.
Loads current JSON from GitHub, adds lat/lon, recommits.
Does NOT overwrite place_name or any other fields.
"""
import requests, json, base64
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}

print("Loading current JSON from GitHub...")
r    = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}", timeout=15)
ravi = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(ravi['lsoas'])} LSOAs | version: {ravi['meta'].get('version')}")

print("\nFetching ONS centroids (geometry extraction)...")
BASE = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_PopCentroids_EW_2021_V4/FeatureServer/0/query")

centroids = {}
offset = 0
while True:
    r = requests.get(BASE, timeout=30, headers=HEADERS, params={
        'where':'1=1','outFields':'LSOA21CD','returnGeometry':'true',
        'outSR':'4326','f':'json','resultOffset':offset,'resultRecordCount':1000,
    })
    data = r.json()
    if 'error' in data: print(f"  Error: {data['error']}"); break
    features = data.get('features',[])
    if not features: break
    for f in features:
        code = f.get('attributes',{}).get('LSOA21CD')
        geom = f.get('geometry',{})
        if code and geom and geom.get('x') and geom.get('y'):
            centroids[code] = {'lat':round(float(geom['y']),5),'lon':round(float(geom['x']),5)}
    offset += len(features)
    if offset%5000==0: print(f"  {offset:,} fetched...")
    if len(features)<1000: break

matched = {c:centroids[c] for c in kent_codes if c in centroids}
print(f"  Matched: {len(matched)}/{len(kent_codes)} Kent LSOAs")

updated = 0
for lsoa in ravi['lsoas']:
    if lsoa['lsoa_code'] in matched:
        lsoa['lat'] = matched[lsoa['lsoa_code']]['lat']
        lsoa['lon'] = matched[lsoa['lsoa_code']]['lon']
        updated += 1

ravi['meta']['coord_method'] = 'ONS population-weighted centroids — geometry extraction'
ravi['meta']['generated']    = datetime.now(timezone.utc).isoformat()
print(f"  Updated: {updated} LSOAs with precise coordinates")

thanet = [l for l in ravi['lsoas'] if l['district']=='Thanet'][:2]
print(f"\nThanet check (expect ~51.3-51.4, 1.26-1.42):")
for l in thanet: print(f"  {l['lsoa_name']}: {l['lat']}, {l['lon']}")

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit(ravi, GITHUB_FILE, GITHUB_TOKEN, f"RAVI — {today} — precise centroids ({updated} LSOAs)")
print(f"Cell 3 done — run Cell 4 for place names")


## Cell 4 — Settlement Place Names
Maps each LSOA to its built-up area name using ONS LSOA to BUA lookup.
Loads current JSON, adds place_name only, recommits.
**Run after Cell 3. Never overwrites coordinates.**

In [ ]:
"""
RAVI Cell 4 — Add ONS settlement place names.
Loads current JSON from GitHub, adds place_name field, recommits.
Does NOT overwrite lat/lon or any other fields.
"""
import requests, json, base64, io
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}

print("Loading current JSON from GitHub...")
r    = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}", timeout=15)
ravi = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(ravi['lsoas'])} LSOAs | coords: {ravi['meta'].get('coord_method','?')[:40]}")

# Verify coordinates are precise before proceeding
sample_lat = ravi['lsoas'][0].get('lat', 0)
if sample_lat in (51.27, 0.0):
    print("\nWARNING: Coordinates look like grid placeholders.")
    print("Run Cell 3 first to add precise coordinates, then re-run this cell.")

print("\nFetching ONS LSOA to Built-Up Area lookup...")
# Confirmed service URL from data.gov.uk metadata
BASE = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA11_BUASD11_BUA11_LAD11_RGN11_EW_LU_133d9317ea7245d288ba6e5d22f0a131/"
        "FeatureServer/0/query")

all_records = {}
offset = 0
batch  = 2000
while True:
    r = requests.get(BASE, timeout=30, headers=HEADERS, params={
        'where':'1=1','outFields':'LSOA11CD,BUASD11NM,BUA11NM',
        'returnGeometry':'false','f':'json',
        'resultOffset':offset,'resultRecordCount':batch,
    })
    print(f"  HTTP {r.status_code} | {len(r.content):,} bytes | offset {offset}")
    if r.status_code!=200: break
    data = r.json()
    if 'error' in data: print(f"  Error: {data['error']}"); break
    features = data.get('features',[])
    if not features: break
    for f in features:
        a = f.get('attributes',{})
        code = a.get('LSOA11CD')
        if code and code in kent_codes:
            all_records[code] = {
                'buasd': str(a.get('BUASD11NM','') or '').strip(),
                'bua':   str(a.get('BUA11NM','')   or '').strip(),
            }
    offset += len(features)
    if len(features)<batch: break

print(f"  Kent LSOAs matched: {len(all_records)}/{len(kent_codes)}")

if len(all_records)==0:
    print("\nLookup unavailable. Download manually from:")
    print("https://geoportal.statistics.gov.uk/datasets/f0aac7ccbfd04cda9eb03e353c613faa/about")
    print("Search: 'LSOA 2011 to Built-up Area Best Fit Lookup' → Download → CSV")
    print("Upload to Colab then run:")
    print("  df = pd.read_csv('your_file.csv')")
    print("  for _, row in df[df.LSOA11CD.isin(kent_codes)].iterrows():")
    print("      all_records[row.LSOA11CD] = {'buasd':str(row.BUASD11NM or ''),'bua':str(row.BUA11NM or '')}")
    raise SystemExit("BUA lookup failed — see instructions above")

def place_name(rec):
    for key in ('buasd','bua'):
        v = rec.get(key,'').strip()
        if v and v not in ('nan','None',''): return v
    return 'Rural (no settlement)'

bua_lookup = {code: place_name(rec) for code, rec in all_records.items()}
n_named = sum(1 for v in bua_lookup.values() if v!='Rural (no settlement)')
n_rural = sum(1 for v in bua_lookup.values() if v=='Rural (no settlement)')
print(f"  Named settlements: {n_named} | Rural: {n_rural}")

# Update ONLY place_name — never touch lat/lon or other fields
for lsoa in ravi['lsoas']:
    lsoa['place_name'] = bua_lookup.get(lsoa['lsoa_code'], 'Unknown')

ravi['meta']['has_place_names'] = True
ravi['meta']['generated']       = datetime.now(timezone.utc).isoformat()

print(f"\n── Top 15 critical/high RAVI LSOAs ──")
print(f"{'LSOA':<25} {'Place':<30} {'District':<22} {'RAVI':>5}")
print("-"*85)
for l in sorted([l for l in ravi['lsoas'] if l['ravi_band'] in ('critical','high')],key=lambda x:-x['ravi'])[:15]:
    print(f"  {l['lsoa_name']:<23} {l['place_name']:<30} {l['district']:<22} {l['ravi']:>5.1f}")

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit(ravi, GITHUB_FILE, GITHUB_TOKEN,
       f"RAVI — {today} — place names ({n_named} settlements, {n_rural} rural)")
print(f"\nDone — all four cells complete")
print(f"View: https://silegrand.github.io/assistivagents/rural-access-vulnerability.html")
